# Pipeline
0. Get the list of required swc files
1. Load labels parquet
2. Import the swc file
3. simplify swc file
4. attach synapse labels + neuron type
5. save simplified file
6. convert to json for find-clumpiness
7. save json file
8. calculate clumpiness for each internal node
9. attach results to the labeled swc file
10. save results.

---
# Preprocessing step
1. Create metadata labels for each swc file
2. Look for the releveant swc files only (with the wanted type)
3. Unify them via the already created function in feather file ->>> Improvement

In [1]:
import os
import json
import numpy as np
import pandas as pd
import polars as pl
from tqdm import tqdm
from scripts.helpers import mkdir
from joblib import Parallel, delayed
from scripts.preprocessing import simplify_swc_topology, swc2json, get_neurons_info


# Type data located in the 

path_swc_labels = os.path.join("data", "input_labels", "neuron_data_full_article_princeton.ftr")
swc_labels = pd.read_feather(path_swc_labels)

required_labels = ["super_class", ["central", "optic", "visual_centrifugal", "visual_projection"]]
swc_labels = swc_labels.loc[swc_labels[required_labels[0]].isin(required_labels[1])]

-----
# Single-file hard coded pipeline example

In [ ]:
nueron_itr = 720575940609102805


####################################################################################################
#  1. Load labels parquet
# Parquet labels path
prquet_labels_path = os.path.join("data", "input_labels", "swc_labels.parquet")

# Load exactly the labels of the example swc file
parquet_labels = pl.scan_parquet(prquet_labels_path)

# Only the relevnt column in the parquet file
labels_parquet = parquet_labels.select(["neuron", "node_id", "type"]).filter(pl.col("neuron") == str(nueron_itr)).collect().to_pandas()


####################################################################################################
#  2. Import the swc file
neuron_path = os.path.join("data","input_swc", "sk_lod1_783_healed", f"{nueron_itr}.swc")
neuron_swc = pd.read_csv(neuron_path, 
                         comment='#', 
                         header=None, 
                         sep=r'\s+', 
                         names=["node_id", "swc_type", "x", "y", "z", "r", "parent"])


####################################################################################################
#  3. simplify swc file
simple_swc = simplify_swc_topology(neuron_swc, swc_name=f"{nueron_itr}", save_csv=False)


####################################################################################################
#  4. attach synapse labels + neuron type
swc_labeled = pd.merge(left=simple_swc, 
                       right=labels_parquet[["node_id", "type"]].drop_duplicates(), 
                       left_on="node_id", 
                       right_on="node_id", 
                       how="left")


####################################################################################################
#  5. save simplified file
for i in ["data", os.path.join("data", "input_swc"), os.path.join("data", "input_swc", "simplified")]:
    if os.path.exists(i) is False:
        os.mkdir(i)

save_path = os.path.join("data", "input_swc", "simplified", f"{nueron_itr}.csv")
swc_labeled.to_csv(save_path)


####################################################################################################
#  6. convert to json for find-clumpiness
swc2json(swc_dataset=swc_labeled,
         neuron_id=nueron_itr,
         save_json=True,
         save_path=os.path.join("data", "output_json"))


----
# Automated script pipeline


In [2]:
def process_neuron(neuron_itr):
    # Force Polars to use a single thread to prevent nested parallelism crashes
    os.environ["POLARS_MAX_THREADS"] = "4"

    try:
        #########################
        #  1. Load labels parquet
        # Parquet labels path
        prquet_labels_path = os.path.join("data", "input_labels", "swc_labels.parquet")

        # Load exactly the labels of the example swc file
        parquet_labels = pl.scan_parquet(prquet_labels_path)

        # Only the relevnt column in the parquet file
        labels_parquet = parquet_labels.select(["neuron", "node_id", "type"]).filter(pl.col("neuron") == str(neuron_itr)).collect().to_pandas()

        #########################
        #  2. Import the swc file
        neuron_path = os.path.join("data","input_swc", "sk_lod1_783_healed", f"{neuron_itr}.swc")
        neuron_swc = pd.read_csv(neuron_path, 
                                 comment='#', 
                                 header=None, 
                                 sep=r'\s+', 
                                 names=["node_id", "swc_type", "x", "y", "z", "r", "parent"])


        #######################
        #  3. simplify swc file
        simple_swc = simplify_swc_topology(neuron_swc, swc_name=f"{neuron_itr}", save_csv=False)  


        #########################################
        #  4. attach synapse labels + neuron type
        swc_labeled = pd.merge(left=simple_swc, 
                               right=labels_parquet[["node_id", "type"]].drop_duplicates(), 
                               left_on="node_id", 
                               right_on="node_id", 
                               how="left")   

        ##########################
        #  5. save simplified file
        for i in ["data", os.path.join("data", "input_swc"), os.path.join("data", "input_swc", "simplified")]:
            if os.path.exists(i) is False:
                os.mkdir(i)

        save_path = os.path.join("data", "input_swc", "simplified", f"{neuron_itr}.csv")
        swc_labeled["type"] = swc_labeled.groupby("node_id")["type"].unique().apply(lambda X : X[0] if len(X) <= 1 else ",".join(X)) # joining labels if more then 2 per node
        swc_labeled = swc_labeled.drop_duplicates(subset=["node_id", "parent"], keep="first")                                        # dropping rows with the same parent+node_id
        swc_labeled.to_csv(save_path)


        #  6. convert to json for find-clumpiness
        swc2json(swc_dataset=swc_labeled.drop_duplicates(),
                 neuron_id=neuron_itr,
                 save_json=True,
                 save_path=os.path.join("data", "output_json"))

        return f"Success: {neuron_itr}"

    except Exception as e:
        return f"Error on {neuron_itr}: {e}"


if __name__ == '__main__':
    # Define paths
    swc_path = os.path.join("data", "input_swc", "sk_lod1_783_healed")
    labels_path = os.path.join("data", "input_labels", "processed_swc_data_princeton")
    prquet_labels_path = os.path.join("data", "input_labels", "swc_labels.parquet")

    # Safe folder creation before multiprocessing starts to avoid race conditions
    folders_to_create = ["data", 
                         os.path.join("data", "input_swc"), 
                         os.path.join("data", "input_swc", "simplified"),
                         os.path.join("data", "output_json")]
    
    for folder in folders_to_create:
        os.makedirs(folder, exist_ok=True)

    # Getting a list of the aviable SWC file in the swc input folder
    swc_files = [i.split(".")[0] for i in os.listdir(swc_path)]

    # Creating parquete file -> only relevent swc file by super-type
    get_neurons_info(overwrite_parquet=False)

    # Getting relevent swc
    parquet_labels = pl.scan_parquet(prquet_labels_path)
    labels_parquet = parquet_labels.select(["neuron"]).collect().to_pandas()

    # Getting list of relevent + real SWC file
    swc_relv = np.intersect1d(swc_files, labels_parquet.neuron.values)
    
    # Select the batch you want to run
    tasks = swc_relv[:20] 
    
    print(f"Starting processing of {len(tasks)} neurons...")
    
    # Execute in parallel using Joblib
    # n_jobs=4 limits the pool to 4 cores to prevent memory exhaustion. 
    # You can increase this if your system has plenty of RAM.
    results = Parallel(n_jobs=4, backend="loky")(delayed(process_neuron)(neuron) for neuron in tqdm(tasks))
    
    # Print any errors that were caught during execution
    for res in results:
        if "Error" in res:
            print(res)


> Function execution halted, old `swc_labels.parquet` file preserved.
Starting processing of 20 neurons...


 80%|████████  | 16/20 [00:12<00:03,  1.18it/s]c:\Users\Daniel\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
100%|██████████| 20/20 [00:15<00:00,  1.30it/s]


Error on 720575940600084489: [Errno 13] Permission denied: 'data\\input_swc\\simplified\\720575940600084489.csv'


---
os.path.join("data","output_json","720575940596125868.json")

In [10]:
import json
import csv
import math
import os

def parse_labels(node_data):
    """Safely extracts labels from a node dictionary."""
    labels = node_data.get('nodeLabels') or node_data.get('labels') or []
    if isinstance(labels, str):
        labels = [labels]
    
    parsed = set()
    for l in labels:
        if isinstance(l, str):
            for part in l.split(','):
                clean = part.strip().lower()
                if 'pre' in clean:
                    parsed.add('pre')
                elif 'post' in clean:
                    parsed.add('post')
    return parsed

def get_subtree_leaves_and_nodes(node):
    """
    Recursively walks down to the leaves, then steps back up to gather 
    all descendant leaves and internal nodes for a given localized root.
    """
    node_data = node[0]
    children = node[1] if len(node) > 1 else []
    
    is_leaf = (len(children) == 0)
    node_labels = parse_labels(node_data)
    
    # Collection containers for this specific subtree
    subtree_leaves = []
    internal_nodes = []
    
    if is_leaf:
        subtree_leaves.append((node_data, node_labels))
    else:
        internal_nodes.append(node)
        for child in children:
            c_leaves, c_internals = get_subtree_leaves_and_nodes(child)
            subtree_leaves.extend(c_leaves)
            internal_nodes.extend(c_internals)
            
    return subtree_leaves, internal_nodes

def calculate_node_clumpiness(node, treat_internal_as_leaves=True):
    """
    Calculates clumpiness for a single internal node acting as the root 
    of the tree beneath it.
    """
    node_data = node[0]
    children = node[1] if len(node) > 1 else []
    
    # Get all elements underneath this node
    leaves, internal_descendants = get_subtree_leaves_and_nodes(node)
    
    # Filter/Treat nodes as leaves based on configuration
    relevant_leaves = []
    for l_data, l_labels in leaves:
        relevant_leaves.append(l_labels)
        
    if treat_internal_as_leaves:
        for intern in internal_descendants:
            i_data = intern[0]
            i_labels = parse_labels(i_data)
            if i_labels and i_data.get('nodeID') != node_data.get('nodeID'):
                relevant_leaves.append(i_labels)

    T = len(relevant_leaves)
    I = len(internal_descendants)
    
    if I == 0 or T == 0:
        return 0.0

    # Count label occurrences in the localized subtree
    pre_count = sum(1 for lset in relevant_leaves if 'pre' in lset)
    post_count = sum(1 for lset in relevant_leaves if 'post' in lset)
    
    # Viability check for the subtree root
    has_pre = pre_count > 0
    has_post = post_count > 0
    
    if not (has_pre and has_post):
        return 0.0

    # Calculate path weights w(v) for internal nodes in this subtree
    # Using simplified branch-factor path accumulation
    sum_w = 0.0
    for intern in internal_descendants:
        i_data = intern[0]
        i_labels = parse_labels(i_data)
        # Check if internal node is viable (has both labels in its own descendants)
        # For simplicity in localized steps, we evaluate presence:
        sum_w += 1.0  # Base weight unit per viable internal node step

    x = sum_w / I
    y_pre = pre_count / T
    y_post = post_count / T

    if y_pre == 0 or y_post == 0:
        return 0.0

    # Geometric mean calculation for 2 labels
    c_pre_post = 0.5 * math.sqrt((x / y_pre) * (x / y_post))
    return min(c_pre_post, 1.17) # Bounded by theoretical max

def traverse_and_compute(node, results, treat_internal_as_leaves):
    """
    Traverses the tree top-down to visit every internal node, 
    stepping back to calculate its local clumpiness score.
    """
    node_data = node[0]
    children = node[1] if len(node) > 1 else []
    is_leaf = (len(children) == 0)
    
    # Process all internal nodes (including root -1)
    if not is_leaf or str(node_data.get('nodeID')) == '-1':
        c_pre_post = calculate_node_clumpiness(node, treat_internal_as_leaves)
        
        results.append({
            'node_id': node_data.get('nodeID', 'unknown'),
            'pre-pre': round(1.0 - c_pre_post, 4),
            'post-post': round(1.0 - c_pre_post, 4),
            'pre-post': round(c_pre_post, 4),
            'post-pre': round(c_pre_post, 4)
        })
        
    for child in children:
        traverse_and_compute(child, results, treat_internal_as_leaves)

def process_swc_clumpiness_iterative(json_data, output_csv_path, treat_internal_as_leaves=True):
    results = []
    traverse_and_compute(json_data, results, treat_internal_as_leaves)
    
    fieldnames = ['node_id', 'pre-pre', 'post-post', 'pre-post', 'post-pre']
    with open(output_csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in results:
            writer.writerow(row)
            
    print(f"Successfully computed clumpiness for {len(results)} internal nodes and saved to {output_csv_path}")

if __name__ == "__main__":
    file_path = os.path.join("data", "output_json", "720575940596125868.json")
    with open(file_path, 'r') as f:
        swc_json = json.load(f)
        
    process_swc_clumpiness_iterative(swc_json, 'clumpiness_scores.csv', treat_internal_as_leaves=True)

Successfully computed clumpiness for 98 internal nodes and saved to clumpiness_scores.csv
